# MNPS Job Classification Evaluation – Severity-Weighted Confusion Analysis v2.0

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/yourrepo/blob/main/MNPS_Eval_SeverityWeighted_v2.ipynb)

This notebook evaluates MNPS job classification runs against **ground truth** and aligns
the evaluation metrics with the **same severity index** used in the
`MNPS_Likelihood_Scorer_v6_SeverityWeighted` analysis notebook.

It assumes you have an evaluation file (CSV) that includes, for each record:

- Ground-truth major role group (e.g., `true_major_group`)
- Predicted major role group (e.g., `pred_major_group`)
- `similarity_score` (KSAC similarity between prediction and ground-truth group/role)
- `salary_amount` (annual salary for the role being evaluated)
- `correction_hours` (estimated time to correct a classification error)
- Optionally: `likelihood_score`, `accuracy_equivalent`, confidence, etc.

This notebook will:

1. Compute the **severity-weighted cost index** per record using the SAME formula as the
   severity-weighted likelihood analysis notebook:
   - KSAC dissimilarity (1 - similarity_score),
   - Salary (higher salary → higher potential financial impact),
   - Time-to-correct (higher hours → higher operational impact).
2. Produce **standard confusion matrices** (counts) and **severity-weighted confusion matrices**
   (sum of severity cost per (true, predicted) pair).
3. Compute summary cost metrics (total severity cost of misclassifications, per-class cost, etc.).
4. Save the augmented evaluation CSV and confusion matrices back to a timestamped run folder.


In [ ]:
# 1️⃣ Setup and Imports

import os
import io
import datetime as dt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Detect Colab
try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['axes.grid'] = True

print(f"Running in Colab: {IN_COLAB}")

In [ ]:
# 2️⃣ (Optional) Mount Google Drive and Set Run Folder

if IN_COLAB:
    drive.mount('/content/drive')
    BASE_RUN_PATH = '/content/drive/MyDrive/MNPS_Eval_Runs'
else:
    BASE_RUN_PATH = os.path.abspath('./MNPS_Eval_Runs')

os.makedirs(BASE_RUN_PATH, exist_ok=True)

run_id = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
run_results_path = os.path.join(BASE_RUN_PATH, f'Run_{run_id}')
os.makedirs(run_results_path, exist_ok=True)

print(f"✅ Evaluation results will be saved under: {run_results_path}")

In [ ]:
# 3️⃣ Configuration – Severity Weights and Column Names

# The severity index is identical to MNPS_Likelihood_Scorer_v6_SeverityWeighted:
#   severity_cost_index = (1 - similarity_score) * (0.7 * salary_norm + 0.3 * hours_norm)

SEVERITY_SALARY_WEIGHT = 0.7
SEVERITY_TIME_WEIGHT   = 0.3

# Column names for true/predicted labels (edit here if your file uses different names)
TRUE_LABEL_COL      = 'true_major_group'     #@param {type:'string'}
PRED_LABEL_COL      = 'pred_major_group'     #@param {type:'string'}

# Optional: If you also want to look at minor subgroups, you can point these to columns as well.
TRUE_SUBGROUP_COL   = ''                     #@param {type:'string'}
PRED_SUBGROUP_COL   = ''                     #@param {type:'string'}

print("✅ Configuration loaded.")
print(f"True label column: {TRUE_LABEL_COL}")
print(f"Pred label column: {PRED_LABEL_COL}")

In [ ]:
# 4️⃣ Load Evaluation CSV (Ground Truth + Predictions)

eval_df = None

LOAD_MODE = 'Upload'  #@param ['Upload', 'Google Drive Path'] {type:'string'}

if LOAD_MODE == 'Upload':
    if not IN_COLAB:
        raise RuntimeError('Upload mode is only available in Colab. Set LOAD_MODE="Google Drive Path" when running locally.')
    print('📂 Please upload your evaluation CSV (e.g., Sample JDs with predictions).')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file uploaded.')
    fname = list(uploaded.keys())[0]
    eval_df = pd.read_csv(io.BytesIO(uploaded[fname]))
    print(f'✅ Loaded {fname} with {len(eval_df)} records.')
else:
    input_path = '/content/drive/MyDrive/MNPS_Eval_Runs/latest/eval_input.csv'  #@param {type:'string'}
    if not os.path.exists(input_path):
        raise FileNotFoundError(f'File not found: {input_path}')
    eval_df = pd.read_csv(input_path)
    print(f'✅ Loaded {input_path} with {len(eval_df)} records.')

assert eval_df is not None, "eval_df failed to load"
print(eval_df.head(5))

In [ ]:
# 5️⃣ Severity Index Helpers – Aligned with Likelihood v6 Notebook

def normalize_column(df: pd.DataFrame, col_name: str):
    if col_name is None or col_name not in df.columns:
        return None, 0.0
    max_val = float(df[col_name].max())
    if max_val <= 0 or np.isnan(max_val):
        return col_name, 0.0
    return col_name, max_val


# Identify salary and hours columns by heuristic if not explicitly named
salary_col = None
for c in eval_df.columns:
    cl = c.lower()
    if 'salary_amount' in cl or (cl.startswith('salary') and 'score' not in cl):
        salary_col = c
        break

hours_col = None
for c in eval_df.columns:
    cl = c.lower()
    if 'correction_hours' in cl or 'hours_to_correct' in cl or 'time_to_correct' in cl:
        hours_col = c
        break

if salary_col is None:
    print('⚠️ Could not automatically identify a salary column; severity will ignore salary.')
if hours_col is None:
    print('⚠️ Could not automatically identify a correction-hours column; severity will ignore hours.')


salary_col, max_salary = normalize_column(eval_df, salary_col)
hours_col,  max_hours  = normalize_column(eval_df, hours_col)

print(f"Using salary column: {salary_col} (max={max_salary})")
print(f"Using hours column:  {hours_col} (max={max_hours})")


def compute_severity_cost(row, max_salary, max_hours):
    """Compute severity index for a single record.

    severity_cost_index = dissimilarity * (w_salary * salary_norm + w_time * hours_norm)

    where:
      dissimilarity = 1 - similarity_score  (clamped 0–1)
      salary_norm   = salary_amount / max_salary   (0–1, if salary column exists)
      hours_norm    = correction_hours / max_hours (0–1, if hours column exists)
    """
    sim = row.get('similarity_score', np.nan)
    if np.isnan(sim):
        sim = 0.0
    sim = min(max(sim, 0.0), 1.0)
    dissimilarity = 1.0 - sim

    # Normalized salary
    if salary_col is not None and max_salary > 0:
        sal_raw = row.get(salary_col, 0.0)
        sal_norm = min(max(sal_raw / max_salary, 0.0), 1.0)
    else:
        sal_norm = 0.0

    # Normalized hours
    if hours_col is not None and max_hours > 0:
        hrs_raw = row.get(hours_col, 0.0)
        hrs_norm = min(max(hrs_raw / max_hours, 0.0), 1.0)
    else:
        hrs_norm = 0.0

    severity_component = (
        SEVERITY_SALARY_WEIGHT * sal_norm +
        SEVERITY_TIME_WEIGHT   * hrs_norm
    )

    return dissimilarity * severity_component


# Apply severity index
eval_df['severity_cost_index'] = eval_df.apply(lambda r: compute_severity_cost(r, max_salary, max_hours), axis=1)

# Severity bands via quantiles (as in v6)
def categorize_severity_band(series: pd.Series) -> pd.Series:
    if series.empty:
        return pd.Series(index=series.index, dtype=object)

    q50 = series.quantile(0.50)
    q75 = series.quantile(0.75)
    q90 = series.quantile(0.90)

    def _band(val):
        if np.isnan(val):
            return 'Unknown'
        if val <= q50:
            return 'Low'
        if val <= q75:
            return 'Medium'
        if val <= q90:
            return 'High'
        return 'Extreme'

    return series.apply(_band)


eval_df['severity_band'] = categorize_severity_band(eval_df['severity_cost_index'])

print("✅ Severity index computed and severity_band assigned.")
print(eval_df[['severity_cost_index', 'severity_band']].head(10))

In [ ]:
# 6️⃣ Confusion Matrices – Counts and Severity-Weighted

if TRUE_LABEL_COL not in eval_df.columns or PRED_LABEL_COL not in eval_df.columns:
    raise ValueError(f"True/pred label columns not found: {TRUE_LABEL_COL}, {PRED_LABEL_COL}")

# Standard count confusion matrix
confusion_counts = pd.crosstab(eval_df[TRUE_LABEL_COL], eval_df[PRED_LABEL_COL], dropna=False)

# Severity-weighted confusion matrix (sum of severity per (true, pred))
confusion_severity = pd.crosstab(
    eval_df[TRUE_LABEL_COL],
    eval_df[PRED_LABEL_COL],
    values=eval_df['severity_cost_index'],
    aggfunc='sum',
    dropna=False
).fillna(0.0)

print("Standard Confusion Matrix (Counts):")
display(confusion_counts)
print("\nSeverity-Weighted Confusion Matrix (Sum of severity_cost_index):")
display(confusion_severity)

In [ ]:
# 7️⃣ Misclassification Cost Metrics

# Identify correct vs misclassified
is_correct = eval_df[TRUE_LABEL_COL] == eval_df[PRED_LABEL_COL]
is_error   = ~is_correct

total_records = len(eval_df)
total_errors  = int(is_error.sum())
total_severity_error = float(eval_df.loc[is_error, 'severity_cost_index'].sum())
avg_severity_error   = float(eval_df.loc[is_error, 'severity_cost_index'].mean()) if total_errors > 0 else 0.0

print(f"Total records: {total_records}")
print(f"Total errors:  {total_errors}")
print(f"Total severity-weighted error cost: {total_severity_error:.3f}")
print(f"Average severity cost per error:    {avg_severity_error:.3f}")


# Per-true-class severity cost of misclassifications
error_df = eval_df[is_error].copy()
severity_by_true = error_df.groupby(TRUE_LABEL_COL)['severity_cost_index'].agg(['count', 'sum', 'mean']).rename(
    columns={'count': 'error_count', 'sum': 'severity_sum', 'mean': 'severity_mean'}
)

print("\nSeverity-weighted error metrics by TRUE class:")
display(severity_by_true)


# Optional: per-confusion-pair normalized cost (mean severity per (true,pred))
confusion_severity_mean = pd.crosstab(
    error_df[TRUE_LABEL_COL],
    error_df[PRED_LABEL_COL],
    values=error_df['severity_cost_index'],
    aggfunc='mean',
    dropna=False
).fillna(0.0)

print("\nMean severity_cost_index for each misclassification pair (true → predicted):")
display(confusion_severity_mean)

In [ ]:
# 8️⃣ Visualizations – Severity Distribution and Per-Class Cost

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# (1) Severity distribution
ax = axes[0]
ax.hist(eval_df['severity_cost_index'], bins=12, alpha=0.7)
ax.set_title('Distribution of Severity Cost Index')
ax.set_xlabel('severity_cost_index')
ax.set_ylabel('Frequency')

# (2) Per-true-class severity sum (errors only)
ax = axes[1]
if not severity_by_true.empty:
    severity_by_true['severity_sum'].sort_values(ascending=False).plot(kind='bar', ax=ax)
    ax.set_title('Total Severity Cost of Misclassifications by TRUE Class')
    ax.set_xlabel('True Class')
    ax.set_ylabel('Severity Sum')
else:
    ax.text(0.5, 0.5, 'No misclassifications', ha='center', va='center')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# 9️⃣ Save Augmented Evaluation CSV and Confusion Matrices

eval_out_path = os.path.join(run_results_path, 'eval_with_severity_index.csv')
eval_df.to_csv(eval_out_path, index=False)
print(f"💾 Saved evaluation file with severity index to: {eval_out_path}")

counts_path = os.path.join(run_results_path, 'confusion_counts.csv')
confusion_counts.to_csv(counts_path)
print(f"💾 Saved confusion counts matrix to: {counts_path}")

severity_path = os.path.join(run_results_path, 'confusion_severity_weighted.csv')
confusion_severity.to_csv(severity_path)
print(f"💾 Saved severity-weighted confusion matrix to: {severity_path}")